# A2-02: Image Segmentation with U-Net

Having covered object detection (bounding boxes), we now move to **image segmentation** — assigning a class label to *every pixel* in the image.

We will build **U-Net** from scratch and train it on the Oxford-IIIT Pet dataset.


## Types of Image Segmentation

Object detection gives us bounding boxes — but what if we need to know exactly *which pixels* belong to each object? That's where segmentation comes in.

There are two fundamentally different segmentation tasks:

| | Semantic Segmentation | Instance Segmentation |
|---|---|---|
| **Question** | What class is each pixel? | Which *individual object* does each pixel belong to? |
| **Output** | Class label per pixel | Object ID + class per pixel |
| **Same-class objects** | Merged together | Separated |
| **Typical model** | U-Net, FCN, DeepLab | Mask R-CNN |

<img src="img/segmentation_types.png" title="Semantic vs Instance Segmentation" style="width: 900px;" />


### Visual comparison

```
Original image: 🐱 🐱 🐶

Semantic segmentation:
  [cat][cat][cat][cat][dog][dog]   ← all cats same color, dog different
   ████████████████  ████████
   cat (label=1)     dog (label=2)

Instance segmentation:
  [cat#1][cat#1][cat#2][cat#2][dog#1]
   ██████  ██████  ████████
   cat id=1  cat id=2  dog id=1
```

**Key insight**: Semantic segmentation cannot distinguish between two cats sitting side by side. Instance segmentation can.

## Semantic Segmentation: FCN and U-Net

### Fully Convolutional Network (FCN, 2015)

The first major deep learning approach to semantic segmentation. The key idea: replace the final FC layers of a classification CNN with **1×1 convolutions**, then **upsample** back to input resolution.

```
Classification CNN:   Conv → Pool → ... → FC → softmax
FCN:                  Conv → Pool → ... → 1×1 Conv → Upsample → pixel labels
```

Problem: upsampling from a very small feature map loses spatial detail — edges and fine boundaries get blurry.

### U-Net (2015)

U-Net solved the spatial detail problem with **skip connections** — direct paths from encoder to decoder.

<img src="img/unet_arch.png" title="U-Net Architecture" style="width: 800px;" />

**Reading the diagram:**

- **Left side (encoder / contracting path)** — standard CNN: DoubleConv blocks (blue arrows) shrink spatial size and double channels at each level via MaxPool (red arrow). Four stages: 64 → 128 → 256 → 512 channels.
- **Bottom (bottleneck)** — deepest representation: 1024 channels at the smallest spatial size. No skip connection here — purely compressed features.
- **Right side (decoder / expanding path)** — up-conv 2×2 (green arrow) doubles spatial size and halves channels at each level. Then concatenate the **skip connection** from the matching encoder stage (gray arrow = "copy and crop"), then DoubleConv to process the merged features.
- **Gray arrows (copy and crop)** — this is the skip connection. The encoder's high-resolution feature map is copied directly to the decoder at the same depth level. "Crop" because the original U-Net had no padding — feature maps shrink slightly through each conv, so the encoder map is center-cropped to match the decoder size. In modern implementations (with `padding=1`) the sizes match exactly — no cropping needed.
- **Top right (output)** — final Conv 1×1 (teal arrow) maps from 64 channels to the number of classes (2 in the original paper for binary segmentation).

**Why do skip connections matter?**

> "Classification only needs to know *what*. Segmentation needs to know *what* AND *exactly where*. Deep features = what. Shallow features = where. Skip connections bring both to the decoder simultaneously."

Without skip connections the decoder must reconstruct spatial detail from the bottleneck alone — like drawing a precise boundary from a blurry thumbnail. With skip connections the decoder has the encoder's full-resolution edge maps to guide it.

### Loss and Metrics

**Per-pixel Cross-Entropy Loss**:
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{ic} \log(\hat{p}_{ic})$$

**mean IoU (mIoU)** — standard metric for segmentation:
$$\text{mIoU} = \frac{1}{C} \sum_{c=1}^{C} \frac{TP_c}{TP_c + FP_c + FN_c}$$

where $TP_c$ = pixels correctly predicted as class $c$, $FP_c$ = pixels wrongly predicted as $c$, $FN_c$ = pixels of class $c$ missed.


## Instance Segmentation: Mask R-CNN (2017)

**Paper:** He et al., *Mask R-CNN*, ICCV 2017

Mask R-CNN extends Faster R-CNN by adding a third output head: a **pixel-level mask** for each detected instance.

### Architecture

```
Input Image
    ↓
CNN Backbone (ResNet + FPN)
    ↓
RPN → Region Proposals
    ↓
ROI Align  ← key improvement over ROI Pooling
    ├── Classification head  → class label
    ├── Bounding box head    → refined (x, y, w, h)
    └── Mask head            → 28×28 binary mask (per class)
```

### ROI Align vs ROI Pooling

ROI Pooling **quantizes** the proposal coordinates to integer pixel positions before pooling — this introduces misalignment that degrades mask quality.

ROI Align uses **bilinear interpolation** to sample features at exact (non-integer) positions, preserving spatial precision:

```
ROI Pooling:    proposal (10.7, 20.3) → snapped to (10, 20) → pool
ROI Align:      proposal (10.7, 20.3) → bilinear interp at (10.7, 20.3) → pool
```

### Mask Head

A small FCN applied independently to each ROI: 4 conv layers → predict a $K \times 28 \times 28$ mask (one binary mask per class $K$). During inference, only the mask for the predicted class is used.

### Loss

$$\mathcal{L} = \mathcal{L}_{cls} + \mathcal{L}_{box} + \mathcal{L}_{mask}$$

$\mathcal{L}_{mask}$ is **binary cross-entropy** applied only to the ground-truth class mask — so classes don't compete with each other.

### When to use which?

| Scenario | Best choice |
|---|---|
| Classify every pixel (road, sky, building) | Semantic — U-Net / DeepLab |
| Count/separate individual objects | Instance — Mask R-CNN |
| Both: per-pixel labels AND separate instances | Panoptic segmentation |

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Dataset: Oxford-IIIT Pet Dataset

**Oxford-IIIT Pet** (Parkhi et al., 2012) — 37 breeds of cats and dogs with pixel-level masks:
- **Class 1**: Pet (foreground)
- **Class 2**: Background
- **Class 3**: Border/uncertain region

**Reference:** Parkhi et al. (2012). *Cats and Dogs*. CVPR.
Dataset: https://www.robots.ox.ac.uk/~vgg/data/pets/

---

In [ ]:
from torchvision.datasets import OxfordIIITPet

os.makedirs('./data', exist_ok=True)
IMG_SIZE = 128

train_raw = OxfordIIITPet('./data', split='trainval', target_types='segmentation', download=True)
test_raw  = OxfordIIITPet('./data', split='test',     target_types='segmentation', download=True)
print(f'Train: {len(train_raw)} | Test: {len(test_raw)}')

class PetSegDataset(Dataset):
    def __init__(self, base, size=128):
        self.ds = base
        self.img_tf  = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
        self.mask_tf = transforms.Compose([
            transforms.Resize((size, size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor(),
        ])
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        img, mask = self.ds[idx]
        img  = self.img_tf(img)
        mask = (self.mask_tf(mask).squeeze(0).long() - 1).clamp(0, 2)
        return img, mask

train_data   = PetSegDataset(train_raw, IMG_SIZE)
test_data    = PetSegDataset(test_raw,  IMG_SIZE)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_data,  batch_size=16, shuffle=False, num_workers=2)

# Visualize samples
mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
CLASS_COLORS = np.array([[255,100,100],[100,100,255],[255,255,100]], dtype=np.uint8)
CLASS_NAMES  = ['Pet', 'Background', 'Border']

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(3):
    img, mask = train_data[i*100]
    img_d  = torch.clamp(img * std + mean, 0, 1).permute(1,2,0).numpy()
    mask_d = CLASS_COLORS[mask.numpy()]
    axes[i][0].imshow(img_d);  axes[i][0].set_title('Image');   axes[i][0].axis('off')
    axes[i][1].imshow(mask_d); axes[i][1].set_title('Mask');    axes[i][1].axis('off')
    axes[i][2].imshow(img_d);  axes[i][2].imshow(mask_d, alpha=0.5)
    axes[i][2].set_title('Overlay'); axes[i][2].axis('off')
patches = [plt.Rectangle((0,0),1,1,color=CLASS_COLORS[i]/255) for i in range(3)]
fig.legend(patches, CLASS_NAMES, loc='lower center', ncol=3)
plt.suptitle('Oxford Pet — Image + Segmentation Mask', fontsize=13)
plt.tight_layout(); plt.show()

## Building U-Net from Scratch

The key building block is **DoubleConv**: Conv → BN → ReLU → Conv → BN → ReLU

---

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive Conv2d -> BN -> ReLU blocks. The core building block of U-Net."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),  # same-padding keeps spatial size
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=3, features=[64, 128, 256, 512]):
        super().__init__()

        # ── Encoder (contracting path) ────────────────────────────────────────
        # Each stage: DoubleConv doubles channels, MaxPool halves spatial size
        # features = [64, 128, 256, 512]  →  4 encoder stages
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(ch, f))   # e.g. 3→64, 64→128, ...
            self.pools.append(nn.MaxPool2d(2))         # /2 spatial size
            ch = f

        # ── Bottleneck ────────────────────────────────────────────────────────
        # Deepest point: highest channels (512→1024), smallest spatial size
        # No skip connection here — pure semantic compression
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)  # 512→1024

        # ── Decoder (expanding path) ──────────────────────────────────────────
        # Each stage: up-conv doubles spatial size, then concat skip, then DoubleConv
        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()
        ch = features[-1] * 2   # start from bottleneck output (1024)
        for f in reversed(features):   # 512, 256, 128, 64
            self.upconvs.append(
                nn.ConvTranspose2d(ch, f, kernel_size=2, stride=2)  # 2x upsample
            )
            self.decoders.append(
                DoubleConv(f * 2, f)   # f*2 because we concat skip (f) + upconv output (f)
            )
            ch = f

        # ── Output head ───────────────────────────────────────────────────────
        # 1x1 conv maps final feature map to per-pixel class scores
        self.output = nn.Conv2d(features[0], n_classes, kernel_size=1)  # 64→n_classes

    def forward(self, x):
        # ── Encoder: save skip connections at each stage ──────────────────────
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x)          # DoubleConv: extract features at this resolution
            skips.append(x)     # save for skip connection to the matching decoder stage
            x = pool(x)         # halve spatial size before next stage

        # ── Bottleneck ────────────────────────────────────────────────────────
        x = self.bottleneck(x)

        # ── Decoder: upsample + skip connection + DoubleConv ─────────────────
        for upconv, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x)       # 2x upsample (ConvTranspose2d)

            # Handle size mismatch due to odd input dimensions
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])

            # Skip connection: concat encoder features with decoder features
            # skip carries fine spatial detail (edges, boundaries) from the encoder
            # x    carries semantic context (what object) from the bottleneck
            x = torch.cat([skip, x], dim=1)   # channels: f + f = f*2

            x = dec(x)          # DoubleConv: fuse skip + upsampled features

        # ── Output ────────────────────────────────────────────────────────────
        return self.output(x)   # (N, n_classes, H, W) — same spatial size as input


model = UNet().to(device)
dummy = torch.randn(2, 3, 128, 128).to(device)
out   = model(dummy)
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}  <- same H x W as input')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')


## U-Net with Pretrained ResNet-18 Encoder

Instead of training the encoder from scratch, we replace it with **ResNet-18 pretrained on ImageNet**.

The encoder stages map directly to ResNet-18 layers:

| U-Net Stage | ResNet-18 layer | Output channels | Spatial size (for 128×128 input) |
|---|---|---|---|
| Stage 1 | `layer1` | 64 | 32×32 |
| Stage 2 | `layer2` | 128 | 16×16 |
| Stage 3 | `layer3` | 256 | 8×8 |
| Stage 4 | `layer4` | 512 | 4×4 |

The decoder is the same as the scratch U-Net — up-conv + skip connection + DoubleConv at each stage.

**Why does this help?**
ResNet-18 was trained to classify 1,000 ImageNet categories — its early layers already detect edges, textures, and shapes. Fine-tuning on Oxford Pet takes far fewer epochs to converge than training from scratch.


In [ ]:
import torchvision.models as models

class UNetResNet18(nn.Module):
    """U-Net with pretrained ResNet-18 encoder. Decoder is identical to scratch U-Net."""

    def __init__(self, n_classes=3, pretrained=True):
        super().__init__()

        # ── Encoder: ResNet-18 pretrained on ImageNet ─────────────────────────
        weights = 'IMAGENET1K_V1' if pretrained else None
        resnet  = models.resnet18(weights=weights)

        # ResNet-18 stem: Conv7x7 + BN + ReLU + MaxPool → (N, 64, H/4, W/4)
        self.stem    = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)

        # Four residual stages — each halves spatial size and doubles channels
        self.enc1 = resnet.layer1   # (N, 64,  H/4,  W/4)
        self.enc2 = resnet.layer2   # (N, 128, H/8,  W/8)
        self.enc3 = resnet.layer3   # (N, 256, H/16, W/16)
        self.enc4 = resnet.layer4   # (N, 512, H/32, W/32)

        # ── Bottleneck ────────────────────────────────────────────────────────
        self.bottleneck = DoubleConv(512, 1024)

        # ── Decoder: same structure as scratch U-Net ──────────────────────────
        # up-conv halves channels and doubles spatial size
        # DoubleConv input = f*2 because we concat skip (f) + upconv output (f)
        self.up4   = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4  = DoubleConv(512 + 512, 512)   # skip from enc4 (512) + up4 (512)

        self.up3   = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3  = DoubleConv(256 + 256, 256)   # skip from enc3 (256) + up3 (256)

        self.up2   = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2  = DoubleConv(128 + 128, 128)   # skip from enc2 (128) + up2 (128)

        self.up1   = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1  = DoubleConv(64 + 64, 64)      # skip from enc1 (64)  + up1 (64)

        # ── Final upsample back to input resolution ───────────────────────────
        # stem downsampled by 4 (Conv7x7 stride-2 + MaxPool stride-2)
        # we need one more 2x upsample to go from H/4 back to H
        self.up0   = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec0  = DoubleConv(32, 32)

        # ── Output head ───────────────────────────────────────────────────────
        self.output = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        # ── Encoder ───────────────────────────────────────────────────────────
        s0 = self.stem(x)    # (N, 64,  H/4,  W/4)  — after stem (no skip here)
        s1 = self.enc1(s0)   # (N, 64,  H/4,  W/4)  — skip 1
        s2 = self.enc2(s1)   # (N, 128, H/8,  W/8)  — skip 2
        s3 = self.enc3(s2)   # (N, 256, H/16, W/16) — skip 3
        s4 = self.enc4(s3)   # (N, 512, H/32, W/32) — skip 4

        # ── Bottleneck ────────────────────────────────────────────────────────
        x = self.bottleneck(s4)   # (N, 1024, H/32, W/32)

        # ── Decoder: upsample → concat skip → DoubleConv ─────────────────────
        x = self.up4(x);  x = self._cat(x, s4);  x = self.dec4(x)  # →H/16
        x = self.up3(x);  x = self._cat(x, s3);  x = self.dec3(x)  # →H/8
        x = self.up2(x);  x = self._cat(x, s2);  x = self.dec2(x)  # →H/4
        x = self.up1(x);  x = self._cat(x, s1);  x = self.dec1(x)  # →H/4 (same as s1)
        x = self.up0(x);  x = self.dec0(x)                          # →H/2 ... →H

        return self.output(x)   # (N, n_classes, H, W)

    def _cat(self, x, skip):
        """Concat skip connection, handling any size mismatch from odd dimensions."""
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:])
        return torch.cat([skip, x], dim=1)


# ── Sanity check ──────────────────────────────────────────────────────────────
resnet_model = UNetResNet18(n_classes=3, pretrained=True).to(device)
dummy = torch.randn(2, 3, 128, 128).to(device)
out   = resnet_model(dummy)
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}   <- same H x W as input')
print(f'Params: {sum(p.numel() for p in resnet_model.parameters()):,}')
scratch_params = sum(p.numel() for p in UNet().parameters())
print(f'UNet scratch params: {scratch_params:,}')


## Training U-Net

Loss: **Cross-Entropy** per pixel

Metric: **mIoU** (mean Intersection over Union) — standard segmentation benchmark

$$\text{IoU} = \frac{\text{Predicted} \cap \text{Ground Truth}}{\text{Predicted} \cup \text{Ground Truth}}$$

---

In [ ]:
# ── Choose which model to train ───────────────────────────────────────────────
# 'scratch'   → UNet built from random init (custom Conv encoder)
# 'resnet18'  → UNet with pretrained ResNet-18 encoder (ImageNet)
MODEL_TYPE = 'resnet18'   # change to 'scratch' to train from scratch

if MODEL_TYPE == 'scratch':
    model = UNet(n_classes=3).to(device)
    print("Training U-Net from scratch")
else:
    model = UNetResNet18(n_classes=3, pretrained=True).to(device)
    print("Training U-Net with pretrained ResNet-18 encoder")

from tqdm import tqdm

def compute_iou(pred, target, n_classes=3):
    pred = pred.argmax(dim=1)
    ious = []
    for cls in range(n_classes):
        inter = ((pred==cls) & (target==cls)).sum().float()
        union = ((pred==cls) | (target==cls)).sum().float()
        if union > 0: ious.append((inter/union).item())
    return np.mean(ious) if ious else 0.0

In [ ]:
EPOCHS    = 5
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

save_name = f'unet_{MODEL_TYPE}_pet.pt'

train_losses, val_ious = [], []
for epoch in range(EPOCHS):
    model.train()
    ep_loss = []
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        imgs, masks = imgs.to(device), masks.to(device)
        loss = criterion(model(imgs), masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep_loss.append(loss.item())
    model.eval()
    ep_iou = []
    with torch.no_grad():
        for imgs, masks in test_loader:
            ep_iou.append(compute_iou(model(imgs.to(device)), masks.to(device)))
    scheduler.step()
    train_losses.append(np.mean(ep_loss))
    val_ious.append(np.mean(ep_iou))
    print(f'Epoch {epoch+1:02d} | Loss: {train_losses[-1]:.4f} | mIoU: {val_ious[-1]:.4f}')

torch.save(model.state_dict(), save_name)
print(f'Saved → {save_name}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, marker='o', color='steelblue')
axes[0].set_title(f'Training Loss ({MODEL_TYPE})'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)
axes[1].plot(val_ious, marker='s', color='darkorange')
axes[1].set_title(f'Validation mIoU ({MODEL_TYPE})'); axes[1].set_xlabel('Epoch'); axes[1].grid(True)
plt.tight_layout(); plt.show()
print(f'Best mIoU: {max(val_ious):.4f}')

## Visualize Predictions

---

In [ ]:
model.load_state_dict(torch.load(save_name, map_location=device))
model.eval()

fig, axes = plt.subplots(5, 4, figsize=(14, 18))
for ax, t in zip(axes[0], ['Input','Ground Truth','Prediction','Overlay']):
    ax.set_title(t, fontsize=11, fontweight='bold')

for row in range(5):
    img, mask = test_data[row*50]
    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
    img_d  = torch.clamp(img * std + mean, 0, 1).permute(1,2,0).numpy()
    axes[row][0].imshow(img_d)
    axes[row][1].imshow(CLASS_COLORS[mask.numpy()])
    axes[row][2].imshow(CLASS_COLORS[pred_mask])
    axes[row][3].imshow(img_d); axes[row][3].imshow(CLASS_COLORS[pred_mask], alpha=0.5)
    for ax in axes[row]: ax.axis('off')

patches = [plt.Rectangle((0,0),1,1,color=CLASS_COLORS[i]/255) for i in range(3)]
fig.legend(patches, CLASS_NAMES, loc='lower center', ncol=3)
plt.suptitle(f'U-Net Results ({MODEL_TYPE}) — Oxford Pet Dataset', fontsize=13)
plt.tight_layout(); plt.show()


# Exercises

## Exercise 1

1. **Skip Connections — Do They Matter?**

   Both models use a **pretrained ResNet-18 encoder** (ImageNet weights). The only difference is whether the decoder uses skip connections or not.

   a) Train **two variants** using the `run.py` script:

   ```bash
   # With skip connections (baseline)
   python3 run.py --model unet_resnet18         --dataset oxford_pet --epochs 20 --train

   # Without skip connections (ablation — same ResNet-18 encoder, skip connections removed)
   python3 run.py --model unet_resnet18_no_skip --dataset oxford_pet --epochs 20 --train
   ```

   b) Compare mIoU:

   | Model | Encoder | Skip connections | Val mIoU | Time/epoch |
   |---|---|---|---|---|
   | U-Net + ResNet-18 | ResNet-18 (ImageNet) | ✅ Yes | ? | ? |
   | U-Net + ResNet-18 (no skip) | ResNet-18 (ImageNet) | ❌ No | ? | ? |

   c) Why do skip connections help segmentation more than they would help classification?

   d) Which skip connection level do you think hurts the most when removed — the first (64ch, highest resolution) or the last (512ch, lowest resolution)? Why?
   
---

## Submission

Submit your work to GitHub. Your repository should contain:

### 1. Training Script (`run.py`)

There are **2 models** to train — same ResNet-18 encoder, skip connections differ:

| Model flag | Encoder | Skip connections | Pretrained |
|---|---|---|---|
| `unet_resnet18` | ResNet-18 | ✅ Yes | ✅ ImageNet |
| `unet_resnet18_no_skip` | ResNet-18 | ❌ No | ✅ ImageNet |

```bash
# 1. Baseline — ResNet-18 encoder + skip connections
python3 run.py --model unet_resnet18         --dataset oxford_pet --epochs 20 --train

# 2. Ablation — same ResNet-18 encoder, skip connections REMOVED
python3 run.py --model unet_resnet18_no_skip --dataset oxford_pet --epochs 20 --train

# Evaluate saved model
python3 run.py --model unet_resnet18 --weights unet_resnet18_pet.pt --dataset oxford_pet --evaluate
```

### 2. `README.md`

Your `README.md` must include:

**Commands used** (exact commands you ran)

**Results table:**

| Model | Encoder | Skip connections | Val mIoU | Time/epoch |
|---|---|---|---|---|
| `unet_resnet18` | ResNet-18 (ImageNet) | ✅ | ? | ? |
| `unet_resnet18_no_skip` | ResNet-18 (ImageNet) | ❌ | ? | ? |

**Discussion** (3–5 sentences): How much did skip connections improve mIoU? When would you choose U-Net over Mask R-CNN?
